<a href="https://colab.research.google.com/github/alishadaudi/eBookstore-client-project/blob/main/Powell's_eBookstore_Pycaret.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Code snippet 1
# Installing pycaret for ML modeling
!pip install git+https://github.com/pycaret/pycaret.git

In [ ]:
# Code snippet 2
# Importing regression components from pycaret
from pycaret.regression import *

In [ ]:
# Code snippet 3
# Importing the dataset
import pandas as pd
data = pd.read_csv('https://docs.google.com/spreadsheets/d/1XSBE7VwTJ-xB300r2JAwZesaAbGTLvMgS4qRxXfDjys/pub?gid=0&single=true&output=csv')

# Display the data
data

,Customer ID,Title,First Name,Middle Name,Last Name,Suffix,Street Address1,Street Address2,City-ZipCode-State,Birth Date,...,Occupation,Gender,Marital Status,Home Owner Status,Number of Cars Owned,Number of Children At Home,Total Number of Children,Annual Income,Avg Monthly Spend,eBook Subscriber Flag
0,11000,NaN,Jon,V,Yang,NaN,3761 N. 14th St,NaN,Cleveland-44101-Ohio,4/8/66,...,Professional,M,M,1,0,0,2,137947,89,0
1,11001,NaN,Eugene,L,Huang,NaN,2243 W St.,NaN,Seattle-98101-Washington,5/14/65,...,Professional,M,S,0,1,3,3,101141,117,1
2,11002,NaN,Ruben,NaN,Torres,NaN,5844 Linden Land,NaN,Omaha-68101-Nebraska,8/12/65,...,Professional,M,M,1,1,3,3,91945,123,0
3,11003,NaN,Christy,NaN,Zhu,NaN,1825 Village Pl.,NaN,Fort Worth-76101-Texas,2/15/68,...,Professional,F,S,0,1,0,0,86688,50,0
4,11004,NaN,Elizabeth,NaN,Johnson,NaN,7553 Harness Circle,NaN,Oakland-94601-California,8/8/68,...,Professional,F,S,1,4,5,5,92771,95,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16514,13121,NaN,Latasha,E,Jimenez,NaN,7259 Birchwood,NaN,San Antonio-78201-Texas,3/22/65,...,Professional,F,M,1,4,5,5,101542,101,0
16515,26100,NaN,James,Garner,Ptaszynski,NaN,7345 Kenwal Rd.,NaN,Pittsburgh-15201-Pennsylvania,4/2/36,...,Professional,F,S,1,2,0,3,46549,46,0
16516,11328,NaN,Julian,M,Griffin,NaN,7398 Withersed Lane,NaN,Honolulu-96801-Hawaii,1/1/40,...,Management,M,M,1,2,0,5,133053,79,0
16517,23077,NaN,Marvin,A,Hernandez,NaN,1019 Book Road,NaN,Anaheim-92801-California,10/20/46,...,Skilled Manual,M,M,1,2,0,4,31930,65,0


In [ ]:
# Code snippet 4
import datetime
import pandas as pd

# Only process if 'Birth Date' column exists
if 'Birth Date' in data.columns:
    # Convert 'Birth Date' to datetime objects, explicitly specifying the format to handle two-digit years
    data['Birth Date'] = pd.to_datetime(data['Birth Date'], format='%m/%d/%y')

    # Correct years that are in the future (e.g., 2066 interpreted from '66' needs to be 1966)
    now = datetime.datetime.now()
    future_birth_year_mask = data['Birth Date'].dt.year > now.year
    data.loc[future_birth_year_mask, 'Birth Date'] -= pd.DateOffset(years=100)

    # Calculate age using the corrected birth dates
    data['Age'] = now.year - data['Birth Date'].dt.year

    # Adjust age if the birth month/day hasn't occurred yet this year
    data.loc[
        (now.month < data['Birth Date'].dt.month) |
        ((now.month == data['Birth Date'].dt.month) & (now.day < data['Birth Date'].dt.day)),
        'Age'
    ] -= 1

    # Drop the original 'Birth Date' column
    data = data.drop(columns=['Birth Date'])

# Display the updated DataFrame with the new 'Age' column (or current state if 'Birth Date' was already processed)
display(data.head())

,Customer ID,Title,First Name,Middle Name,Last Name,Suffix,Street Address1,Street Address2,City-ZipCode-State,Education Level,...,Gender,Marital Status,Home Owner Status,Number of Cars Owned,Number of Children At Home,Total Number of Children,Annual Income,Avg Monthly Spend,eBook Subscriber Flag,Age
0,11000,NaN,Jon,V,Yang,NaN,3761 N. 14th St,NaN,Cleveland-44101-Ohio,Bachelors,...,M,M,1,0,0,2,137947,89,0,60
1,11001,NaN,Eugene,L,Huang,NaN,2243 W St.,NaN,Seattle-98101-Washington,Bachelors,...,M,S,0,1,3,3,101141,117,1,60
2,11002,NaN,Ruben,NaN,Torres,NaN,5844 Linden Land,NaN,Omaha-68101-Nebraska,Bachelors,...,M,M,1,1,3,3,91945,123,0,60
3,11003,NaN,Christy,NaN,Zhu,NaN,1825 Village Pl.,NaN,Fort Worth-76101-Texas,Bachelors,...,F,S,0,1,0,0,86688,50,0,58
4,11004,NaN,Elizabeth,NaN,Johnson,NaN,7553 Harness Circle,NaN,Oakland-94601-California,Bachelors,...,F,S,1,4,5,5,92771,95,1,57


In [ ]:
# Code snippet 5

# Split the 'City-ZipCode-State' column into three new columns
data[['City', 'Zip Code', 'State']] = data['City-ZipCode-State'].str.split('-', expand=True)

# Drop the original 'City-ZipCode-State' column
data = data.drop(columns=['City-ZipCode-State'])

# Display the updated DataFrame to verify the changes
display(data.head())

,Customer ID,Title,First Name,Middle Name,Last Name,Suffix,Street Address1,Street Address2,Education Level,Occupation,...,Number of Cars Owned,Number of Children At Home,Total Number of Children,Annual Income,Avg Monthly Spend,eBook Subscriber Flag,Age,City,Zip Code,State
0,11000,NaN,Jon,V,Yang,NaN,3761 N. 14th St,NaN,Bachelors,Professional,...,0,0,2,137947,89,0,60,Cleveland,44101,Ohio
1,11001,NaN,Eugene,L,Huang,NaN,2243 W St.,NaN,Bachelors,Professional,...,1,3,3,101141,117,1,60,Seattle,98101,Washington
2,11002,NaN,Ruben,NaN,Torres,NaN,5844 Linden Land,NaN,Bachelors,Professional,...,1,3,3,91945,123,0,60,Omaha,68101,Nebraska
3,11003,NaN,Christy,NaN,Zhu,NaN,1825 Village Pl.,NaN,Bachelors,Professional,...,1,0,0,86688,50,0,58,Fort Worth,76101,Texas
4,11004,NaN,Elizabeth,NaN,Johnson,NaN,7553 Harness Circle,NaN,Bachelors,Professional,...,4,5,5,92771,95,1,57,Oakland,94601,California


In [ ]:
# Code snippet 6
# Displaying dataset columns
data.columns

Index(['Customer ID', 'Title', 'First Name', 'Middle Name', 'Last Name',
       'Suffix', 'Street Address1', 'Street Address2', 'Education Level',
       'Occupation', 'Gender', 'Marital Status', 'Home Owner Status',
       'Number of Cars Owned', 'Number of Children At Home',
       'Total Number of Children', 'Annual Income', 'Avg Monthly Spend',
       'eBook Subscriber Flag', 'Age', 'City', 'Zip Code', 'State'],
      dtype='object')

In [ ]:
# Code snippet 7
# Setting the target variable
y = 'Avg Monthly Spend'

In [ ]:
# Code snippet 6
# Setting the ignored columns list
ignored_cols = ['Customer ID', 'Title', 'First Name', 'Middle Name', 'Last Name',
                'Suffix', 'Street Address1', 'Street Address2', 'eBook Subscriber Flag']

In [ ]:
# Code snippet 7
# Setting the categorical columns list
cat_cols = ['Education Level', 'Occupation', 'Gender',
       'Marital Status', 'Home Owner Status',
            'City', 'Zip Code', 'State']

In [ ]:
# Code snippet 8
# Setting the numerical columns list
num_cols  = ['Number of Cars Owned', 'Number of Children At Home',
             'Total Number of Children', 'Annual Income', 'Age']

In [ ]:
# Code snippet 9
# Setting/configuring ML setup with pycaret
regression_setup = setup(data,
                        target = y,
                        ignore_features = ignored_cols,
                        categorical_features = cat_cols,
                        numeric_features = num_cols)

,Description,Value
0,Session id,3019
1,Target,Avg Monthly Spend
2,Target type,Regression
3,Original data shape,"(16519, 23)"
4,Transformed data shape,"(16519, 22)"
5,Transformed train set shape,"(11563, 22)"
6,Transformed test set shape,"(4956, 22)"
7,Ignore features,9
8,Numeric features,5
9,Categorical features,8


In [ ]:
# Code snippet 10
# Invoking ML algorithms
compare_models()

,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
lightgbm,Light Gradient Boosting Machine,2.4929,10.0001,3.1608,0.9868,0.0505,0.0387,0.4460
gbr,Gradient Boosting Regressor,2.5432,10.3274,3.2130,0.9864,0.0509,0.0392,1.1610
xgboost,Extreme Gradient Boosting,2.6288,11.0024,3.3157,0.9855,0.0528,0.0408,0.2590
rf,Random Forest Regressor,2.6621,11.3971,3.3747,0.9850,0.0537,0.0413,3.9950
et,Extra Trees Regressor,2.7792,12.4339,3.5251,0.9836,0.0563,0.0432,2.6890
dt,Decision Tree Regressor,3.5774,20.6578,4.5430,0.9727,0.0727,0.0555,0.3140
lr,Linear Regression,4.8653,41.3902,6.4317,0.9454,0.0921,0.0706,0.5760
ridge,Ridge Regression,4.8645,41.3903,6.4317,0.9454,0.0920,0.0706,0.1380
br,Bayesian Ridge,4.8647,41.3903,6.4317,0.9454,0.0920,0.0706,0.1860
lar,Least Angle Regression,4.8798,41.6250,6.4503,0.9451,0.0922,0.0708,0.1450


Processing:   0%|          | 0/81 [00:00<?, ?it/s]

LGBMRegressor(n_jobs=-1, random_state=3019)

In [ ]:
# Code snippet 11
# Performing cross validation (running the best algorithm 10 times)
best_model = create_model('lightgbm')

,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,2.4156,9.5602,3.0920,0.9883,0.0479,0.0367
1,2.4665,9.6630,3.1085,0.9860,0.0512,0.0391
2,2.5646,11.1558,3.3400,0.9860,0.0557,0.0404
3,2.4474,9.4943,3.0813,0.9876,0.0473,0.0370
4,2.4374,9.5634,3.0925,0.9872,0.0495,0.0379
5,2.4805,9.8301,3.1353,0.9869,0.0502,0.0387
6,2.4795,9.6209,3.1018,0.9870,0.0480,0.0378
7,2.6184,11.2623,3.3559,0.9850,0.0531,0.0408
8,2.4885,9.7076,3.1157,0.9875,0.0493,0.0385


Processing:   0%|          | 0/4 [00:00<?, ?it/s]

In [ ]:
# Code snippet 12
# Importing the client dataset(for which the target values/avg monthly spend needs to be predicted)
client_data = pd.read_csv('https://docs.google.com/spreadsheets/d/16s5TlD28DsgScenFfe7cMrzdCl75LQZA3mbOM_EekxA/pub?gid=1509803897&single=true&output=csv')

client_data

,CustomerID,Title,FirstName,MiddleName,LastName,Suffix,StreetAddress1,StreetAddress2,City-ZipCode-State,BirthDate,EducationLevel,Occupation,Gender,MaritalStatus,HomeOwnerStatus,NumCarsOwned,NumChildrenAtHome,TotalNumChildren,AnnualIncome
0,Customer_xx_1289,NaN,Trevor,NaN,Jenkins,NaN,4697 Yosemite Dr.,NaN,Seattle-98101-Washington,2/27/1953,Graduate Degree,Management,M,S,1,2,0,3,101674
1,Customer_xx_1290,NaN,Zachary,NaN,Anderson,NaN,2476 Mt. Whitney Way,NaN,Oklahoma City-73101-Oklahoma,5/14/1939,Bachelors,Management,M,M,1,2,0,5,131143
2,Customer_xx_1291,NaN,Frank,F,Navarro,NaN,Buergermeister-ulrich-str 2987,NaN,Tampa-33601-Florida,8/10/1961,Graduate Degree,Clerical,M,M,1,0,0,1,58637
3,Customer_xx_1292,NaN,Dennis,M,She,NaN,"828, rue de Berri",NaN,Charlotte-28201-North Carolina,7/8/1961,Graduate Degree,Clerical,M,S,1,0,0,1,45259
4,Customer_xx_1293,NaN,Randy,A,Zeng,NaN,2334 Brandywine Way,NaN,Plano-75023-Texas,8/21/1936,Graduate Degree,Clerical,M,M,1,0,0,1,46868
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2603,Customer_xx_3892,NaN,Cynthia,NaN,Kapoor,NaN,7045 Creekside Drive,NaN,Denver-80201-Colorado,7/28/1978,Partial College,Manual,F,S,0,1,0,0,20689
2604,Customer_xx_3893,NaN,Lance,NaN,Jimenez,NaN,Auf Der Steige 532,NaN,Miami-33101-Florida,11/26/1943,Bachelors,Clerical,M,M,1,0,0,1,59132
2605,Customer_xx_3894,NaN,Dominique,M,Saunders,NaN,3792 Westwood Ct.,NaN,Fresno-93701-California,4/26/1961,Graduate Degree,Clerical,F,M,0,0,0,3,65669
2606,Customer_xx_3895,NaN,Clayton,NaN,Zhang,NaN,"1080, quai de Grenelle",NaN,Corpus Christi-78401-Texas,3/5/1959,Bachelors,Clerical,M,M,1,0,0,3,59736


In [ ]:
# Code snippet 13
import datetime
import pandas as pd # Ensure pandas is imported if not globally available

# Clean up column names by stripping whitespace
client_data.columns = client_data.columns.str.strip()

# Rename columns in client_data to match training data's expected format
client_data = client_data.rename(columns={
    'EducationLevel': 'Education Level',
    'MaritalStatus': 'Marital Status',
    'HomeOwnerStatus': 'Home Owner Status',
    'NumCarsOwned': 'Number of Cars Owned',
    'NumChildrenAtHome': 'Number of Children At Home',
    'TotalNumChildren': 'Total Number of Children',
    'AnnualIncome': 'Annual Income'
})

# Apply the same 'Birth Date' to 'Age' transformation as in cell 17i7_37rnb8i
# Only process if 'BirthDate' column exists
if 'BirthDate' in client_data.columns:
    # Convert 'BirthDate' to datetime objects, explicitly specifying the format to handle two-digit years
    client_data['BirthDate'] = pd.to_datetime(client_data['BirthDate'], format='%m/%d/%Y') # Assuming 4-digit year in client_data

    # Calculate age using the corrected birth dates
    now = datetime.datetime.now()
    client_data['Age'] = now.year - client_data['BirthDate'].dt.year

    # Adjust age if the birth month/day hasn't occurred yet this year
    client_data.loc[
        (now.month < client_data['BirthDate'].dt.month) |
        ((now.month == client_data['BirthDate'].dt.month) & (now.day < client_data['BirthDate'].dt.day)),
        'Age'
    ] -= 1

    # Drop the original 'BirthDate' column
    client_data = client_data.drop(columns=['BirthDate'])


# Apply the same 'City-ZipCode-State' transformation as in cell vms4_tIDnpWW
# Only process if 'City-ZipCode-State' column exists
if 'City-ZipCode-State' in client_data.columns:
    # Split the 'City-ZipCode-State' column into three new columns
    client_data[['City', 'Zip Code', 'State']] = client_data['City-ZipCode-State'].str.split('-', expand=True)

    # Drop the original 'City-ZipCode-State' column
    client_data = client_data.drop(columns=['City-ZipCode-State'])

# Debugging: Print columns in client_data before prediction to verify preprocessing
print("Columns in client_data before prediction:")
print(client_data.columns)
display(client_data.head())

# Predicting avg monthly spend for the client dataset
predictions = predict_model(best_model, client_data)

# Exporting predictions to a CSV file
predictions.to_csv('eBookstore Client Avg Monthly Spend predictions.csv')

Columns in client_data before prediction:
Index(['CustomerID', 'Title', 'FirstName', 'MiddleName', 'LastName', 'Suffix',
       'StreetAddress1', 'StreetAddress2', 'Education Level', 'Occupation',
       'Gender', 'Marital Status', 'Home Owner Status', 'Number of Cars Owned',
       'Number of Children At Home', 'Total Number of Children',
       'Annual Income', 'Age', 'City', 'Zip Code', 'State'],
      dtype='object')


,CustomerID,Title,FirstName,MiddleName,LastName,Suffix,StreetAddress1,StreetAddress2,Education Level,Occupation,...,Marital Status,Home Owner Status,Number of Cars Owned,Number of Children At Home,Total Number of Children,Annual Income,Age,City,Zip Code,State
0,Customer_xx_1289,NaN,Trevor,NaN,Jenkins,NaN,4697 Yosemite Dr.,NaN,Graduate Degree,Management,...,S,1,2,0,3,101674,73,Seattle,98101,Washington
1,Customer_xx_1290,NaN,Zachary,NaN,Anderson,NaN,2476 Mt. Whitney Way,NaN,Bachelors,Management,...,M,1,2,0,5,131143,86,Oklahoma City,73101,Oklahoma
2,Customer_xx_1291,NaN,Frank,F,Navarro,NaN,Buergermeister-ulrich-str 2987,NaN,Graduate Degree,Clerical,...,M,1,0,0,1,58637,64,Tampa,33601,Florida
3,Customer_xx_1292,NaN,Dennis,M,She,NaN,"828, rue de Berri",NaN,Graduate Degree,Clerical,...,S,1,0,0,1,45259,64,Charlotte,28201,North Carolina
4,Customer_xx_1293,NaN,Randy,A,Zeng,NaN,2334 Brandywine Way,NaN,Graduate Degree,Clerical,...,M,1,0,0,1,46868,89,Plano,75023,Texas


In [ ]:
predictions

,CustomerID,Title,FirstName,MiddleName,LastName,Suffix,StreetAddress1,StreetAddress2,Education Level,Occupation,...,Home Owner Status,Number of Cars Owned,Number of Children At Home,Total Number of Children,Annual Income,Age,City,Zip Code,State,prediction_label
0,Customer_xx_1289,NaN,Trevor,NaN,Jenkins,NaN,4697 Yosemite Dr.,NaN,Graduate Degree,Management,...,1,2,0,3,101674,73,Seattle,98101,Washington,76.258093
1,Customer_xx_1290,NaN,Zachary,NaN,Anderson,NaN,2476 Mt. Whitney Way,NaN,Bachelors,Management,...,1,2,0,5,131143,86,Oklahoma City,73101,Oklahoma,78.511813
2,Customer_xx_1291,NaN,Frank,F,Navarro,NaN,Buergermeister-ulrich-str 2987,NaN,Graduate Degree,Clerical,...,1,0,0,1,58637,64,Tampa,33601,Florida,73.982504
3,Customer_xx_1292,NaN,Dennis,M,She,NaN,"828, rue de Berri",NaN,Graduate Degree,Clerical,...,1,0,0,1,45259,64,Charlotte,28201,North Carolina,65.691882
4,Customer_xx_1293,NaN,Randy,A,Zeng,NaN,2334 Brandywine Way,NaN,Graduate Degree,Clerical,...,1,0,0,1,46868,89,Plano,75023,Texas,60.617672
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2603,Customer_xx_3892,NaN,Cynthia,NaN,Kapoor,NaN,7045 Creekside Drive,NaN,Partial College,Manual,...,0,1,0,0,20689,47,Denver,80201,Colorado,38.993492
2604,Customer_xx_3893,NaN,Lance,NaN,Jimenez,NaN,Auf Der Steige 532,NaN,Bachelors,Clerical,...,1,0,0,1,59132,82,Miami,33101,Florida,67.977057
2605,Customer_xx_3894,NaN,Dominique,M,Saunders,NaN,3792 Westwood Ct.,NaN,Graduate Degree,Clerical,...,0,0,0,3,65669,64,Fresno,93701,California,51.775369
2606,Customer_xx_3895,NaN,Clayton,NaN,Zhang,NaN,"1080, quai de Grenelle",NaN,Bachelors,Clerical,...,1,0,0,3,59736,67,Corpus Christi,78401,Texas,74.895885
